# Demonstrating mutagenize_orf()

`mutagenize_orf()` applies codon-level mutations to ORF sequences, supporting multiple biologically-relevant mutation types.

In [1]:
# Basic usage: introduce exactly 1 missense mutation per sequence
import poolparty as pp

# A simple ORF: ATG (M) + AAA (K) + TTT (F) + GGG (G) = 12 bp, 4 codons
orf_seq = 'ATGAAATTTGGG'

with pp.Party() as party:
    # Sequential mode enumerates all possible single missense mutations
    mutants = pp.mutagenize_orf(
        orf_seq, 
        num_mutations=1, 
        mutation_type='missense_only_first',  # Default: different AA, no stop
        mode='sequential'
    ).named('mutant')

print(f"Number of single missense mutants: {mutants.num_states}")
print(f"(4 codons × 19 alternative AAs = 76 mutants)")

df = mutants.generate_library(num_cycles=1)
df[['seq', 'mutant.op.key.codon_positions', 'mutant.op.key.wt_aas', 'mutant.op.key.mut_aas']]

Number of single missense mutants: 76
(4 codons × 19 alternative AAs = 76 mutants)


,seq,mutant.op.key.codon_positions,mutant.op.key.wt_aas,mutant.op.key.mut_aas
0,TTCAAATTTGGG,"(0,)","(M,)","(F,)"
1,CTGAAATTTGGG,"(0,)","(M,)","(L,)"
2,ATCAAATTTGGG,"(0,)","(M,)","(I,)"
3,GTGAAATTTGGG,"(0,)","(M,)","(V,)"
4,AGCAAATTTGGG,"(0,)","(M,)","(S,)"
...,...,...,...,...
71,ATGAAATTTGAC,"(3,)","(G,)","(D,)"
72,ATGAAATTTGAG,"(3,)","(G,)","(E,)"
73,ATGAAATTTTGC,"(3,)","(G,)","(C,)"
74,ATGAAATTTTGG,"(3,)","(G,)","(W,)"


In [2]:
# Different mutation types

# Nonsense mutations (introduce stop codons)
with pp.Party() as party:
    nonsense = pp.mutagenize_orf(
        'ATGAAATTT',  # M-K-F
        num_mutations=1,
        mutation_type='nonsense',
        mode='sequential'
    ).named('nonsense')

print("=== Nonsense mutations (introduce stop codons) ===")
print(f"States: {nonsense.num_states} (3 codons × 3 stop codons = 9)")
df = nonsense.generate_library(num_cycles=1)
print(df[['seq', 'nonsense.op.key.wt_aas', 'nonsense.op.key.mut_aas']].to_string())

# Synonymous mutations (same AA, different codon)
with pp.Party() as party:
    # Use Leucine (CTG) which has 5 synonymous alternatives
    syn = pp.mutagenize_orf(
        'CTGCTGCTG',  # L-L-L
        num_mutations=1,
        mutation_type='synonymous',
        mode='random'
    ).named('syn')

print("\n=== Synonymous mutations (same AA, different codon) ===")
df = syn.generate_library(num_seqs=5, seed=42)
print(df[['seq', 'syn.op.key.wt_codons', 'syn.op.key.mut_codons', 'syn.op.key.wt_aas', 'syn.op.key.mut_aas']].to_string())

=== Nonsense mutations (introduce stop codons) ===
States: 9 (3 codons × 3 stop codons = 9)
         seq nonsense.op.key.wt_aas nonsense.op.key.mut_aas
0  TGAAAATTT                   (M,)                    (*,)
1  TAAAAATTT                   (M,)                    (*,)
2  TAGAAATTT                   (M,)                    (*,)
3  ATGTGATTT                   (K,)                    (*,)
4  ATGTAATTT                   (K,)                    (*,)
5  ATGTAGTTT                   (K,)                    (*,)
6  ATGAAATGA                   (F,)                    (*,)
7  ATGAAATAA                   (F,)                    (*,)
8  ATGAAATAG                   (F,)                    (*,)

=== Synonymous mutations (same AA, different codon) ===
         seq syn.op.key.wt_codons syn.op.key.mut_codons syn.op.key.wt_aas syn.op.key.mut_aas
0  TTACTGCTG               (CTG,)                (TTA,)              (L,)               (L,)
1  CTGTTGCTG               (CTG,)                (TTG,)          

In [3]:
# ORF boundaries: mutate only the coding region, preserve UTRs

# Sequence with 5' UTR (GGGGG) + ORF (ATGAAATTT) + 3' UTR (CCCCC)
full_seq = 'GGGGG' + 'ATGAAATTT' + 'CCCCC'
print(f"Full sequence: {full_seq}")
print(f"               {'5UTR-':>5}{'--ORF---':>9}{'3UTR-':>5}")

with pp.Party() as party:
    mutants = pp.mutagenize_orf(
        full_seq,
        num_mutations=1,
        orf_start=5,   # ORF starts at position 5
        orf_end=14,    # ORF ends at position 14
        mode='random'
    ).named('mutant')

df = mutants.generate_library(num_seqs=5, seed=42)
print("\nMutated sequences (UTRs preserved):")
for seq in df['seq']:
    print(f"  {seq[:5]} | {seq[5:14]} | {seq[14:]}")

Full sequence: GGGGGATGAAATTTCCCCC
               5UTR- --ORF---3UTR-

Mutated sequences (UTRs preserved):
  GGGGG | GAGAAATTT | CCCCC
  GGGGG | ATGGCCTTT | CCCCC
  GGGGG | ATGTGGTTT | CCCCC
  GGGGG | GACAAATTT | CCCCC
  GGGGG | CTGAAATTT | CCCCC


In [4]:
# Codon position selection: restrict which codons can be mutated

# 6-codon ORF
orf_seq = 'ATGAAATTTGGGCCCAAA'  # M-K-F-G-P-K

with pp.Party() as party:
    # Only mutate codons at positions 1, 3, 5 (0-indexed)
    # i.e., skip codons 0 (M), 2 (F), 4 (P)
    mutants = pp.mutagenize_orf(
        orf_seq,
        num_mutations=1,
        codon_positions=[1, 3, 5],  # Only K, G, K can be mutated
        mode='sequential'
    ).named('mutant')

print(f"Eligible codons: positions 1, 3, 5 (K, G, K)")
print(f"Number of states: {mutants.num_states} (3 positions × 19 alternatives = 57)")

# Alternative: use codon_start/codon_end/codon_step_size
with pp.Party() as party:
    # Mutate every other codon starting from position 0
    mutants2 = pp.mutagenize_orf(
        orf_seq,
        num_mutations=1,
        codon_start=0,
        codon_end=6,
        codon_step_size=2,  # Positions 0, 2, 4
        mode='sequential'
    ).named('mutant')

print(f"\nUsing step_size=2: positions 0, 2, 4 (M, F, P)")
print(f"Number of states: {mutants2.num_states}")

Eligible codons: positions 1, 3, 5 (K, G, K)
Number of states: 57 (3 positions × 19 alternatives = 57)

Using step_size=2: positions 0, 2, 4 (M, F, P)
Number of states: 57


In [5]:
# Random mutagenesis with mutation_rate

# 10-codon ORF
orf_seq = 'ATG' * 10  # 30 bp

with pp.Party() as party:
    # Each codon has 30% chance of being mutated
    mutants = pp.mutagenize_orf(
        orf_seq,
        mutation_rate=0.3,
        mutation_type='any_codon',  # Any different codon
        mode='random'
    ).named('mutant')

df = mutants.generate_library(num_seqs=10, seed=42)

print("Random mutagenesis (30% rate per codon):")
print("Number of mutations per sequence:")
for i, row in df.iterrows():
    n_mut = len(row['mutant.op.key.codon_positions'])
    print(f"  Sequence {i}: {n_mut} mutations at positions {list(row['mutant.op.key.codon_positions'])}")

Random mutagenesis (30% rate per codon):
Number of mutations per sequence:
  Sequence 0: 4 mutations at positions [3, 4, 8, 9]
  Sequence 1: 4 mutations at positions [1, 4, 5, 8]
  Sequence 2: 4 mutations at positions [2, 3, 4, 7]
  Sequence 3: 4 mutations at positions [0, 3, 4, 5]
  Sequence 4: 3 mutations at positions [0, 3, 5]
  Sequence 5: 6 mutations at positions [0, 1, 2, 4, 6, 7]
  Sequence 6: 4 mutations at positions [0, 3, 5, 6]
  Sequence 7: 2 mutations at positions [2, 6]
  Sequence 8: 0 mutations at positions []
  Sequence 9: 4 mutations at positions [1, 4, 5, 6]
